# Implémentation des algorithmes d’Arbre de Décision et de Gradient Boosting.

Chargement et préparation du dataset Higgs Boson

In [55]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
#import kagglehub as kb

Chargement du dataset

In [18]:
dataset = pd.read_csv("training.csv")
# Download latest version
#path = kb.competition_download('higgs-boson')

#print("Path to competition files:", path)

FileNotFoundError: [Errno 2] No such file or directory: 'training.csv'

 Suppression des colonnes inutiles

In [ ]:
dataset = dataset.drop(columns=["EventId", "Weight"])

In [ ]:
#Transformation ou conersion des etiquettes : s = signal -> 1,  b = background -> 0
dataset["Label"] = dataset["Label"].map({
    "s": 1,
    "b": 0
})
# Gestion des valeurs manquantes
dataset = dataset.replace(-999.0, np.nan)

# Remplissage par la moyenne
dataset = dataset.fillna(dataset.mean())

# Variables explicatives
X = dataset.drop(columns=["Label"]).values

# Variable cible
y = dataset["Label"].values

Séparation des donnée

In [ ]:
# Mélange aléatoire
np.random.seed(42)
indices = np.random.permutation(len(X))
# 80 % entraînement
taille_entrainements = int(0.8 * len(X))
train_indices = indices[:taille_entrainements]
test_indices = indices[taille_entrainements:]
# Données d'entraînement
X_train = X[train_indices]
y_train = y[train_indices]
# Données de test
X_test = X[test_indices]
y_test = y[test_indices]

Arbre de Décision : Fonction d’impureté de Gini:  IG(D)=1-(pO^2 +p1^2)

In [19]:
def calcul_gini(labels):
    classes = np.unique(labels)
    impurite = 1
    for classe in classes:
        proportion = np.sum(labels == classe) / len(labels)
        impurite -= proportion ** 2
    return impurite

Recherche de la meilleure division: Selon le pseaudo-code CART, on recherche: 
- la meilleur variable,
- le meilleur seuil,
- maximisant le gain d'information

In [26]:
def meilleure_separation(X, y):
    meilleure_variable = None
    meilleur_seuil = None
    meilleur_gain = -1
    gini_parent = calcul_gini(y)
    nombre_variables = X.shape[1]
    for variable in range(nombre_variables):
        seuils = np.unique(X[:, variable])
        for seuil in seuils:
            gauche = X[:, variable] <= seuil
            droite = X[:, variable] > seuil
            if np.sum(gauche) == 0:
                continue
            if np.sum(droite) == 0:
                continue
            y_gauche = y[gauche]
            y_droite = y[droite]
            taille_totale = len(y)
            taille_gauche = len(y_gauche)
            taille_droite = len(y_droite)
            gini_gauche = calcul_gini(y_gauche)
            gini_droite = calcul_gini(y_droite)
            gini_pondere = (
                (taille_gauche / taille_totale) * gini_gauche
                +
                (taille_droite / taille_totale) * gini_droite
            )
            gain = gini_parent - gini_pondere
            if gain > meilleur_gain:
                meilleur_gain = gain
                meilleure_variable = variable
                meilleur_seuil = seuil
    return meilleure_variable, meilleur_seuil

Structure d’un noeud CART

In [28]:
class Noeud:
    def _init_(
        self,
        variable=None,
        seuil=None,
        gauche=None,
        droite=None,
        valeur=None
    ):
        self.variable = variable
        self.seuil = seuil
        self.gauche = gauche
        self.droite = droite
        self.valeur = valeur

Construction récursive de l’Arbre de Décision CART

In [32]:
class ArbreDecision:
    def _init_(self, profondeur_max=4):
        self.profondeur_max = profondeur_max
        self.racine = None
    def entrainer(self, X, y):
        self.racine = self.construire_arbre(X, y)
    def construire_arbre(
        self,
        X,
        y,
        profondeur=0
    ):
        nombre_classes = len(np.unique(y))
        # Condition d'arrêt
        if (
            profondeur >= self.profondeur_max
            or nombre_classes == 1
        ):
             prediction = np.bincount(y).argmax()
            return Noeud(valeur=prediction)
        variable, seuil = meilleure_separation(X, y)
        if variable is None:
            prediction = np.bincount(y).argmax()
            return Noeud(valeur=prediction)
        masque_gauche = X[:, variable] <= seuil
        masque_droite = X[:, variable] > seuil
        branche_gauche = self.construire_arbre(
            X[masque_gauche],
            y[masque_gauche],
            profondeur + 1
        )
        branche_droite = self.construire_arbre(
            X[masque_droite],
            y[masque_droite],
            profondeur + 1
        )
        return Noeud(
            variable,
            seuil,
            branche_gauche,
            branche_droite
        )
    def prediction_individuelle(
        self,
        observation,
        noeud
    ):
        if noeud.valeur is not None:
            return noeud.valeur

        if observation[noeud.variable] <= noeud.seuil:

            return self.prediction_individuelle(
                observation,
                noeud.gauche
            )
        return self.prediction_individuelle(
            observation,
            noeud.droite
        )
    def predire(self, X):
        predictions = []
        for observation in X:
            valeur = self.prediction_individuelle(
                observation,
                self.racine
            )
            predictions.append(valeur)
        return np.array(predictions)

IndentationError: unindent does not match any outer indentation level (<string>, line 20)

In [ ]:
Entraînement de l’Arbre, modèle CART

In [ ]:
modele_arbre = ArbreDecision(
    profondeur_max=5
)
modele_arbre.entrainer(
    X_train,
    y_train
)
predictions_arbre = modele_arbre.predire(X_test)

In [ ]:
#Calcul de la précision
precision_arbre = np.mean(
    predictions_arbre == y_test
)
print(
    "Précision Arbre de Décision :",
    precision_arbre
)
#Accuracy du modèle
accuracy_cart=np.mean(predictions_cart==y_test)
print("accuracy cart: ",accuracy_cart)

Implémentation du Gradient Boosting repose sur :
- Une descente de gradient fonctionnelle
- La correction progressive des erreurs
- les pseudo-résidus

In [57]:
#Régression faible
class ModeleFaible:
    def entrainer(self, X, y):
        self.moyenne = np.mean(y)
    def predire(self, X):
        return np.full(
            len(X),
            self.moyenne
        )
#Implémentation du Gradient Boosting
class GradientBoosting:
    def _init_(
        self,
        nombre_modeles=20,
        taux_apprentissage=0.1
    ):
        self.nombre_modeles = nombre_modeles
        self.taux_apprentissage = taux_apprentissage
        self.modeles = []
    def entrainer(self, X, y):
        prediction = np.zeros(len(y))
        for iteration in range(self.nombre_modeles):
            erreurs = y - prediction
            modele = ModeleFaible()
            modele.entrainer(X, erreurs)
            correction = modele.predire(X)
            prediction += (
                self.taux_apprentissage
                * correction
            )
            self.modeles.append(modele)
    def predire(self, X):
        prediction = np.zeros(len(X))
        for modele in self.modeles:
            prediction += (
                self.taux_apprentissage
                * modele.predire(X)
            )
        return (
            prediction > 0.5
        ).astype(int)

#Entraînement du Gradient Boosting
modele_boosting = GradientBoosting(
    nombre_modeles=30,
    taux_apprentissage=0.1
)

modele_boosting.entrainer(
    X_train,
    y_train
)
predictions_boosting = modele_boosting.predire(
    X_test
)
#Évaluation du modèle
precision_boosting = np.mean(
    predictions_boosting == y_test
)
print(
    "Précision Gradient Boosting :",
    precision_boostingdef matrice_confusion(y_reel, y_prediction):

    vrai_positif = np.sum(
        (y_reel == 1)
        &
        (y_prediction == 1)
    )

    vrai_negatif = np.sum(
        (y_reel == 0)
        &
        (y_prediction == 0)
    )

    faux_positif = np.sum(
        (y_reel == 0)
        &
        (y_prediction == 1)
    )

    faux_negatif = np.sum(
        (y_reel == 1)
        &
        (y_prediction == 0)
    )

    return np.array([
        [vrai_positif, faux_negatif],
        [faux_positif, vrai_negatif]
    ])

print(
    matrice_confusion(
        y_test,
        predictions_arbre
    )
)

print(
    matrice_confusion(
        y_test,
        predictions_boosting
    )
)
an(predictions_boosting == y_test)
print ("Accuracy Gradient Boosting:", accuracy_boosting)

SyntaxError: '(' was never closed (1144571708.py, line 60)

Matrice de confusion

In [37]:
def matrice_confusion(y_reel, y_prediction):
    vrai_positif = np.sum(
        (y_reel == 1)
        &
        (y_prediction == 1)
    )
    vrai_negatif = np.sum(
        (y_reel == 0)
        &
        (y_prediction == 0)
    )
    faux_positif = np.sum(
        (y_reel == 0)
        &
        (y_prediction == 1)
    )
    faux_negatif = np.sum(
        (y_reel == 1)
        &
        (y_prediction == 0)
    )
    return np.array([
        [vrai_positif, faux_negatif],
        [faux_positif, vrai_negatif]
    ])
print(
    matrice_confusion(
        y_test,
        predictions_arbre
    )
)
print(
    matrice_confusion(
        y_test,
        predictions_boosting
    )
)

NameError: name 'y_test' is not defined

Visualisation des résultats

In [40]:
noms_modeles = [
    "Arbre de Décision ou CART",
    "Gradient Boosting"
]
scores = [
    precision_arbre,
    precision_boosting
]
plt.figure(figsize=(8, 5))
plt.bar(
    noms_modeles,
    scores
)
plt.ylabel("Précision")
plt.title(
    "Comparaison des modèles"
)
plt.show()

NameError: name 'precision_arbre' is not defined